# Historical electricity operating scheduler

This notebook demonstrates the usecase of choosing the cheapest uninterrupted operating window using historical spot prices. It utelize the Gold-layer from `data/scheduling/prices` and recommend a start/end time and the estimated spot-energy cost.

## 1. Project setup

In [5]:
import sys
from pathlib import Path
from zoneinfo import ZoneInfo

from pyspark.sql import SparkSession

from gold_layer.operating_scheduler import recommend

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

GOLD_ROOT = PROJECT_ROOT / "data" / "gold"


spark = (
    SparkSession.builder.appName("DAT535 Historical Scheduler Demo")
    .master("local[4]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

prices = spark.read.parquet(str(GOLD_ROOT / "scheduling" / "prices"))
oslo = ZoneInfo("Europe/Oslo")

print("Gold price rows:", prices.count())

Gold price rows: 87374


## 2. Run a scheduling request

The `recommend()` is located in `gold_layer/operating_scheduler.py`. The recommendations are computed from Gold prices.

In [6]:
winter = recommend(
    prices,
    area="NO2",
    local_start="2024-01-15 08:00",
    local_end="2024-01-15 20:00",
    duration_hours=3,
    power_kw=2,
)

Area: NO2 | Allowed: 2024-01-15 08:00 to 2024-01-15 20:00
Device: 2 kW for 3 consecutive hours
Available price slots: 12
Recommended: 2024-01-15 12:00:00+01:00 to 2024-01-15 15:00:00+01:00
Energy: 6 kWh
Estimated spot-energy cost: 5.65 NOK


In [7]:
summer = recommend(
    prices,
    area="NO5",
    local_start="2025-07-15 06:00",
    local_end="2025-07-15 22:00",
    duration_hours=4,
    power_kw=1.5,
)

Area: NO5 | Allowed: 2025-07-15 06:00 to 2025-07-15 22:00
Device: 1.5 kW for 4 consecutive hours
Available price slots: 16
Recommended: 2025-07-15 11:00:00+02:00 to 2025-07-15 15:00:00+02:00
Energy: 6.0 kWh
Estimated spot-energy cost: 0.90 NOK


## 5. Missing data demonstration
Use the known daylight saving time related quarantined period.

In [8]:
missing = recommend(
    prices,
    area="NO1",
    local_start="2024-10-27 08:00",
    local_end="2024-10-27 20:00",
    duration_hours=3,
    power_kw=2,
)

Area: NO1 | Allowed: 2024-10-27 08:00 to 2024-10-27 20:00
Device: 2 kW for 3 consecutive hours
Available price slots: 0
No complete operating window: missing prices or insufficient time.
